# CSE 151B — Full Pipeline (External Math SFT + DPO + GRPO + Self-Consistency)

End-to-end recipe assuming **unlimited compute time**. Ranked by expected gain:

1. **QLoRA SFT on a large external math corpus** (biggest lever for a 4B model).
2. **DPO** on self-sampled (correct, incorrect) pairs from the SFT model.
3. **GRPO** (optional, last) with the project `Judger` as the outcome reward.
4. **Self-consistency** with large N at inference time.
5. Few-shot + CoT (always on) and **progressive-hint** for low-confidence items.
6. Greedy CoT fallback (the implicit baseline).

## Recoverable checkpoints
Every long stage writes incremental JSONL/Trainer checkpoints. Re-running a cell after a crash **resumes** instead of starting over:

| Stage | Resumable artifact |
|---|---|
| External SFT data conversion | `training/external_sft.jsonl` — re-runs skip if file exists |
| QLoRA SFT training | `training/sft_lora/checkpoint-*` (HF Trainer `resume_from_checkpoint=True`) |
| Self-sampling on public set | `training/self_samples.jsonl` — appended per batch, dedup by `(id, sample_idx)` |
| DPO pair construction | `training/dpo_pairs.jsonl` — idempotent rebuild from samples |
| DPO training | `training/dpo_lora/checkpoint-*` |
| GRPO training | `training/grpo_lora/checkpoint-*` |
| Private inference | `results/private_samples.jsonl` — appended per chunk, skip done ids on resume |
| Progressive-hint round | `results/private_hint_samples.jsonl` — same |

## External datasets (all on Hugging Face Hub, free, publicly available)
Listed best-to-worst for this task. The notebook auto-downloads them via `datasets.load_dataset`. You need `huggingface_hub` logged in only if you want gated datasets — none of these require auth.

| HF dataset id | Size | Why | License |
|---|---|---|---|
| `AI-MO/NuminaMath-CoT` | ~860k | High-quality competition-math problems w/ CoT solutions, the strongest single corpus for math reasoning SFT in 2024–25. | Apache-2.0 |
| `nvidia/OpenMathInstruct-2` | ~14M | Massive synthetic math QA w/ solutions distilled from Llama-3.1-405B; broad coverage. | CC-BY-4.0 |
| `meta-math/MetaMathQA` | ~395k | Rephrased GSM8K + MATH; great for arithmetic + word problems. | MIT |
| `TIGER-Lab/MathInstruct` | ~260k | Mix of CoT + program-of-thought; helps with symbolic problems. | MIT |
| `hendrycks/competition_math` | ~12.5k | The MATH benchmark train split; gold-standard solutions. | MIT |
| `openai/gsm8k` (`main` config) | ~7.5k | Grade-school word problems; easy wins. | MIT |
| `nvidia/OpenMathReasoning` | ~3M | Reasoning-traces dataset (uses Qwen/DeepSeek teacher); aligns well with Qwen3-Thinking style. | CC-BY-4.0 |

For a single overnight run start with **NuminaMath-CoT + MetaMathQA + competition_math + gsm8k** (~1.3M rows). Add OpenMathInstruct-2 only if you have multi-day budget.

Run the cells **top to bottom**. Each stage's switch (`RUN_*`) defaults to the recommended setting.

## 0. Environment
Reuses the project `.venv`. First run uncomment the pip line.

In [ ]:
# One-time install (uncomment first run).
# !.venv/bin/python -m pip install -q --upgrade peft trl datasets accelerate sentencepiece
!source ./.venv/bin/activate && echo "venv ready"

## 1. Configuration
Edit the switches and dataset list here. Defaults assume you want every stage on.

In [ ]:
import os, json, re, sys, gc, csv, math, random, time, glob, shutil, hashlib
from pathlib import Path
from typing import Optional, List, Tuple, Iterable
from collections import Counter, defaultdict

BASE_MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"

PUBLIC_PATH      = "data/public.jsonl"
PRIVATE_PATH     = "data/private.jsonl"

RESULTS_DIR      = Path("results");  RESULTS_DIR.mkdir(exist_ok=True)
TRAIN_DIR        = Path("training"); TRAIN_DIR.mkdir(exist_ok=True)

EXT_SFT_PATH         = TRAIN_DIR / "external_sft.jsonl"
SELF_SAMPLES_PATH    = TRAIN_DIR / "self_samples.jsonl"
DPO_PAIRS_PATH       = TRAIN_DIR / "dpo_pairs.jsonl"
GRPO_PROMPTS_PATH    = TRAIN_DIR / "grpo_prompts.jsonl"
SFT_LORA_DIR         = TRAIN_DIR / "sft_lora"
DPO_LORA_DIR         = TRAIN_DIR / "dpo_lora"
GRPO_LORA_DIR        = TRAIN_DIR / "grpo_lora"
MERGED_SFT_DIR       = Path("/home/prgowda/merged_sft")
MERGED_FINAL_DIR     = Path("/home/prgowda/merged_final")
PRIVATE_SAMPLES_PATH = RESULTS_DIR / "private_samples.jsonl"
PRIVATE_HINTS_PATH   = RESULTS_DIR / "private_hint_samples.jsonl"
SUBMISSION_CSV       = Path("submission.csv")

# ── Stage switches ──────────────────────────────────────────────────────────
RUN_BUILD_EXT_SFT    = True
RUN_SFT              = True
RUN_MERGE_SFT        = True
RUN_SELF_SAMPLE      = True
RUN_BUILD_DPO        = True
RUN_DPO              = True
RUN_GRPO             = True
RUN_MERGE_FINAL      = True
RUN_PRIVATE_INFER    = True
RUN_HINT_ROUND       = True

# ── External SFT corpora ────────────────────────────────────────────────────
# Curated: highest-quality math CoT only. MetaMath dropped (mostly redundant w/ GSM8K).
EXT_DATASETS = [
    ("AI-MO/NuminaMath-CoT",        None,   30_000),
    ("hendrycks/competition_math",   None,     None),
    ("openai/gsm8k",                 "main",   None),
]

# ── Hyperparameters ─────────────────────────────────────────────────────────
GPU_ID                = "0"
MAX_MODEL_LEN         = 8192        # inference context
INFER_MAX_MODEL_LEN   = 16384
MAX_NEW_TOKENS_INFER  = 8192
MAX_NEW_TOKENS_SAMPLE = 4096

N_SAMPLES_PER_Q       = 16
N_SAMPLES_SELF        = 6
PROGRESSIVE_HINT_TAU  = 0.5
N_HINT_SAMPLES        = 8
FEW_SHOT_K_MCQ        = 2
FEW_SHOT_K_FREE       = 2
INFER_TEMPERATURE     = 0.7
INFER_TOP_P           = 0.95
INFER_TOP_K           = 20
INFER_CHUNK           = 64

SFT_EPOCHS            = 1
SFT_LR                = 1e-4
SFT_BSZ               = 1
SFT_GRAD_ACCUM        = 8          # was 32; reduced for 24 GB MIG slice
SFT_SAVE_STEPS        = 200
SFT_LOG_STEPS         = 20

DPO_EPOCHS            = 1
DPO_LR                = 5e-6
DPO_BETA              = 0.1

GRPO_EPOCHS           = 1
GRPO_LR               = 5e-6
GRPO_NUM_GENERATIONS  = 4
GRPO_BETA             = 0.04

os.environ["CUDA_VISIBLE_DEVICES"]        = GPU_ID
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
os.environ["TOKENIZERS_PARALLELISM"]      = "false"
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

random.seed(0)
print("Config OK.")

## 2. Load competition data + judger

In [ ]:
sys.path.insert(0, ".")
from judger import Judger
from utils import last_boxed_only_string, remove_boxed
judger = Judger(strict_extract=False)

def load_jsonl(p):
    return [json.loads(l) for l in open(p)]

public_data  = load_jsonl(PUBLIC_PATH)
private_data = load_jsonl(PRIVATE_PATH)
print(f"public={len(public_data)}  private={len(private_data)}")

## 3. Prompt construction (CoT + few-shot + optional progressive hint)

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Reason carefully and step-by-step. "
    "At the very end, output ONLY the final answer wrapped in a single \\boxed{}. "
    "If the problem has multiple [ANS] placeholders, output ALL final answers in order, "
    "comma-separated, inside ONE \\boxed{}, e.g. \\boxed{3, 7, -1}. "
    "Do not add units inside \\boxed{} unless the question explicitly asks. "
    "Simplify radicals and fractions where possible."
)
SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. Read the problem and the answer choices, reason step-by-step, "
    "then output ONLY the letter of the single best option inside \\boxed{}, e.g. \\boxed{C}."
)

def _short(item, lim_q=350, lim_a=80):
    if len(item["question"]) > lim_q: return False
    if item.get("options"):
        return len(str(item.get("answer", ""))) <= 3 and all(len(o) <= lim_a for o in item["options"])
    a = item.get("answer", [])
    if not isinstance(a, list): a = [a]
    return len(a) <= 2 and all(len(str(x)) <= 30 for x in a)

_mcq_short  = [d for d in public_data if d.get("options") and _short(d)]
_free_short = [d for d in public_data if not d.get("options") and _short(d)]
random.Random(7).shuffle(_mcq_short); random.Random(7).shuffle(_free_short)
FEW_SHOT_MCQ  = _mcq_short[:FEW_SHOT_K_MCQ]
FEW_SHOT_FREE = _free_short[:FEW_SHOT_K_FREE]

def _format_choices(options):
    labels = [chr(65 + i) for i in range(len(options))]
    return "\n".join(f"{l}. {o.strip()}" for l, o in zip(labels, options))

def _exemplar_block(items, is_mcq):
    if not items: return ""
    out = ["Worked examples (study the format, then solve the new question):\n"]
    for ex in items:
        q = ex["question"].strip()
        if is_mcq:
            out.append(f"Example question:\n{q}\n\nOptions:\n{_format_choices(ex['options'])}\n\nFinal answer: \\boxed{{{str(ex['answer']).strip().upper()}}}\n")
        else:
            a = ex["answer"] if isinstance(ex["answer"], list) else [ex["answer"]]
            out.append(f"Example question:\n{q}\n\nFinal answer: \\boxed{{{', '.join(str(x) for x in a)}}}\n")
    out.append("Now solve the NEW question below. Show your reasoning then finish with \\boxed{...}.\n")
    return "\n".join(out)

def build_messages(item, hints: Optional[List[str]] = None):
    is_mcq = bool(item.get("options"))
    system = SYSTEM_PROMPT_MCQ if is_mcq else SYSTEM_PROMPT_MATH
    block  = _exemplar_block(FEW_SHOT_MCQ if is_mcq else FEW_SHOT_FREE, is_mcq)
    parts  = [block] if block else []
    parts.append("New question:\n" + item["question"].strip())
    if is_mcq:
        parts.append("\nOptions:\n" + _format_choices(item["options"]))
    if hints:
        parts.append(
            f"\nHint: previous attempts produced the candidate answer(s) {{{', '.join(str(h) for h in hints)}}}. "
            "Reconsider from scratch and either confirm or correct. Final answer inside \\boxed{}."
        )
    return [
        {"role": "system", "content": system},
        {"role": "user",   "content": "\n".join(parts)},
    ]

def chat_template_prompt(tok, messages):
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

## 4. Answer extraction + voting + judging

In [ ]:
_LETTER_RE = re.compile(r"\\boxed\{\s*([A-Za-z])\s*\}")

def extract_letter(text: str) -> str:
    m = _LETTER_RE.search(text or "")
    if m: return m.group(1).upper()
    for line in reversed((text or "").splitlines()):
        ms = re.findall(r"\b([A-Z])\b", line.strip())
        if ms: return ms[-1]
    return ""

def extract_boxed_inner(text: str) -> str:
    if not text: return ""
    b = last_boxed_only_string(text)
    return (remove_boxed(b) or "").strip() if b else ""

def normalize_free_answer(text: str) -> str:
    s = extract_boxed_inner(text)
    if not s: return ""
    for r in (" ", "$", "\\,", "\\;", "\\!"):
        s = s.replace(r, "")
    return s

def vote(responses: List[str], is_mcq: bool) -> Tuple[str, float, str]:
    if not responses: return "", 0.0, ""
    keys = [extract_letter(r) if is_mcq else normalize_free_answer(r) for r in responses]
    c = Counter(k for k in keys if k)
    if not c: return "", 0.0, responses[0]
    top_key, top_n = c.most_common(1)[0]
    cands = [r for r, k in zip(responses, keys) if k == top_key]
    return top_key, top_n / len(responses), max(cands, key=len)

def judge_response(item, response: str) -> bool:
    if bool(item.get("options")):
        return extract_letter(response) == str(item["answer"]).strip().upper()
    gold = item["answer"] if isinstance(item["answer"], list) else [item["answer"]]
    try:
        return bool(judger.auto_judge(pred=response, gold=gold, options=[[]] * len(gold)))
    except Exception:
        return False

## 5. Build the external SFT corpus (checkpointed)

Downloads each HF dataset, converts every row to a `{prompt, completion}` line using the same chat template the model will see at inference, and streams them to `training/external_sft.jsonl`. If the file already exists this cell is a no-op (delete the file to rebuild).

In [ ]:
def _make_freeform_item(question_text: str) -> dict:
    return {"question": question_text, "options": None}

def _row_to_pair(tok, hf_id: str, row: dict) -> Optional[Tuple[str, str]]:
    """Map any of the supported corpora to (prompt, completion)."""
    q, sol = None, None
    if hf_id == "AI-MO/NuminaMath-CoT":
        q   = row.get("problem")
        sol = row.get("solution")
    elif hf_id == "meta-math/MetaMathQA":
        q   = row.get("query")
        sol = row.get("response")
    elif hf_id == "hendrycks/competition_math":
        q   = row.get("problem")
        sol = row.get("solution")
    elif hf_id == "openai/gsm8k":
        q   = row.get("question")
        a   = row.get("answer", "")
        # GSM8K solutions end with "#### N" — rewrite to a boxed final answer.
        if "####" in a:
            reasoning, _, final = a.rpartition("####")
            sol = reasoning.strip() + f"\n\nFinal answer: \\boxed{{{final.strip()}}}"
        else:
            sol = a
    elif hf_id == "nvidia/OpenMathInstruct-2":
        q   = row.get("problem") or row.get("question")
        sol = row.get("generated_solution") or row.get("solution")
        ans = row.get("expected_answer")
        if sol and ans and "\\boxed" not in sol:
            sol = sol.rstrip() + f"\n\nFinal answer: \\boxed{{{ans}}}"
    elif hf_id == "TIGER-Lab/MathInstruct":
        q   = row.get("instruction")
        sol = row.get("output")
    elif hf_id == "nvidia/OpenMathReasoning":
        q   = row.get("problem")
        sol = row.get("generated_solution") or row.get("solution")
    if not q or not sol: return None
    if "\\boxed" not in sol:
        return None  # require a boxed answer so the model learns the output format
    prompt = chat_template_prompt(tok, build_messages(_make_freeform_item(q)))
    return prompt, sol.strip()

if RUN_BUILD_EXT_SFT and not EXT_SFT_PATH.exists():
    from datasets import load_dataset
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    tok.pad_token = tok.eos_token

    n_written = 0
    tmp_path = EXT_SFT_PATH.with_suffix(".jsonl.tmp")
    with open(tmp_path, "w") as f:
        for hf_id, cfg, max_rows in EXT_DATASETS:
            print(f"→ {hf_id} (config={cfg}, cap={max_rows})")
            try:
                ds = load_dataset(hf_id, cfg, split="train", streaming=True)
            except Exception as e:
                print("  skipped:", e); continue
            kept = 0
            for i, row in enumerate(ds):
                if max_rows and i >= max_rows: break
                pair = _row_to_pair(tok, hf_id, row)
                if not pair: continue
                f.write(json.dumps({"prompt": pair[0], "completion": pair[1], "src": hf_id}) + "\n")
                kept += 1; n_written += 1
                if kept % 5000 == 0:
                    print(f"    {kept} rows")
            print(f"  kept {kept}")
    tmp_path.replace(EXT_SFT_PATH)
    print(f"Wrote {n_written} rows -> {EXT_SFT_PATH}")
else:
    print("External SFT corpus:", EXT_SFT_PATH, "present" if EXT_SFT_PATH.exists() else "SKIPPED")

## 6. QLoRA SFT (resumable)

HuggingFace `Trainer` checkpoints every `SFT_SAVE_STEPS` to `training/sft_lora/checkpoint-N`. Re-running this cell after a crash auto-resumes from the latest checkpoint.

In [ ]:
def _latest_ckpt(dir_path: Path) -> Optional[str]:
    ckpts = sorted(glob.glob(str(dir_path / "checkpoint-*")),
                    key=lambda p: int(p.rsplit("-", 1)[-1]))
    return ckpts[-1] if ckpts else None

if RUN_SFT:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, prepare_model_for_kbit_training
    from trl import SFTTrainer, SFTConfig
    from datasets import load_dataset

    assert EXT_SFT_PATH.exists(), "Run section 5 first."
    ds = load_dataset("json", data_files=str(EXT_SFT_PATH), split="train")

    tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    tok.pad_token = tok.eos_token
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                              bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID, quantization_config=bnb, device_map="auto", trust_remote_code=True,
    )
    base = prepare_model_for_kbit_training(base)

    lora_cfg = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                          "gate_proj","up_proj","down_proj"],
    )

    eos = tok.eos_token
    def fmt(ex): return {"text": ex["prompt"] + ex["completion"] + eos}
    ds = ds.map(fmt, remove_columns=[c for c in ds.column_names if c not in ("prompt","completion")])

    cfg = SFTConfig(
        output_dir=str(SFT_LORA_DIR),
        per_device_train_batch_size=SFT_BSZ,
        gradient_accumulation_steps=SFT_GRAD_ACCUM,
        num_train_epochs=SFT_EPOCHS,
        learning_rate=SFT_LR, lr_scheduler_type="cosine", warmup_ratio=0.03,
        bf16=True, logging_steps=SFT_LOG_STEPS,
        save_strategy="steps", save_steps=SFT_SAVE_STEPS, save_total_limit=3,
        max_length=2048, packing=True, report_to="none",
        gradient_checkpointing=True, optim="adamw_8bit",
        dataset_text_field="text",
        use_liger_kernel=True,
    )
    trainer = SFTTrainer(model=base, args=cfg, train_dataset=ds,
                          peft_config=lora_cfg, processing_class=tok)
    resume = _latest_ckpt(SFT_LORA_DIR)
    print("Resuming from", resume) if resume else print("Starting SFT fresh")
    trainer.train(resume_from_checkpoint=resume)
    trainer.save_model(str(SFT_LORA_DIR))
    print("SFT-LoRA saved ->", SFT_LORA_DIR)
    del trainer, base; gc.collect(); torch.cuda.empty_cache()
else:
    print("Skipping SFT.")

## 7. Merge SFT adapter into BF16 weights (for vLLM)

In [ ]:
if RUN_MERGE_SFT and not MERGED_SFT_DIR.exists():
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    print("Merging SFT adapter…")
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID, torch_dtype=torch.bfloat16, device_map="cpu", trust_remote_code=True,
    )
    merged = PeftModel.from_pretrained(base, str(SFT_LORA_DIR)).merge_and_unload()
    merged.save_pretrained(MERGED_SFT_DIR, safe_serialization=True)
    AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True).save_pretrained(MERGED_SFT_DIR)
    del base, merged; gc.collect(); torch.cuda.empty_cache()
    print("Merged ->", MERGED_SFT_DIR)
else:
    print("Merged SFT:", MERGED_SFT_DIR, "present" if MERGED_SFT_DIR.exists() else "skipped")

## 8. Self-sample the SFT model on the public set (checkpointed)

Writes `training/self_samples.jsonl` with one line per `(id, sample_idx)`. Re-running resumes from where it stopped (dedup on `(id, sample_idx)`).

In [ ]:
def _read_done_keys(path: Path, key_fields=("id", "sample_idx")):
    if not path.exists(): return set()
    done = set()
    with open(path) as f:
        for line in f:
            try:
                r = json.loads(line)
                done.add(tuple(r[k] for k in key_fields))
            except Exception: pass
    return done

if RUN_SELF_SAMPLE:
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams
    model_path = str(MERGED_SFT_DIR) if MERGED_SFT_DIR.exists() else BASE_MODEL_ID
    use_bnb = (model_path == BASE_MODEL_ID)
    print("Self-sampling with:", model_path)

    tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    tok.pad_token = tok.eos_token

    done = _read_done_keys(SELF_SAMPLES_PATH)
    have_per_id = Counter(k[0] for k in done)
    todo = [it for it in public_data if have_per_id[it["id"]] < N_SAMPLES_SELF]
    print(f"To sample: {len(todo)}/{len(public_data)} questions (already have {len(done)} traces)")

    if todo:
        kw = dict(model=model_path, trust_remote_code=True,
                   gpu_memory_utilization=0.80, max_model_len=INFER_MAX_MODEL_LEN,
                   max_num_seqs=256, enable_prefix_caching=True)
        if use_bnb: kw.update(quantization="bitsandbytes", load_format="bitsandbytes")
        llm = LLM(**kw)
        sp = SamplingParams(n=N_SAMPLES_SELF, max_tokens=MAX_NEW_TOKENS_SAMPLE,
                             temperature=0.9, top_p=0.95, top_k=40)
        # process in chunks to checkpoint frequently
        for start in range(0, len(todo), INFER_CHUNK):
            batch = todo[start:start+INFER_CHUNK]
            prompts = [chat_template_prompt(tok, build_messages(it)) for it in batch]
            outs = llm.generate(prompts, sampling_params=sp)
            with open(SELF_SAMPLES_PATH, "a") as f:
                for item, out in zip(batch, outs):
                    for i, c in enumerate(out.outputs):
                        if (item["id"], i) in done: continue
                        f.write(json.dumps({
                            "id": item["id"], "sample_idx": i,
                            "is_mcq": bool(item.get("options")),
                            "gold": item.get("answer"),
                            "trace": c.text.strip(),
                        }) + "\n")
            print(f"  flushed up to question {start+len(batch)}/{len(todo)}")
        del llm; gc.collect()
        try:
            import torch; torch.cuda.empty_cache()
        except Exception: pass
else:
    print("Skipping self-sample.")

## 9. Build DPO pairs from the self-samples (idempotent)

For each public question, pair its longest correct trace as `chosen` with its longest incorrect trace as `rejected`.

In [ ]:
if RUN_BUILD_DPO:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    tok.pad_token = tok.eos_token
    id_to_item = {it["id"]: it for it in public_data}
    bucket = defaultdict(list)
    with open(SELF_SAMPLES_PATH) as f:
        for line in f:
            r = json.loads(line); bucket[r["id"]].append(r)
    n_pairs = 0
    with open(DPO_PAIRS_PATH, "w") as out:
        for qid, rows in bucket.items():
            item = id_to_item.get(qid)
            if not item: continue
            good, bad = [], []
            for r in rows:
                (good if judge_response(item, r["trace"]) else bad).append(r["trace"])
            if not good or not bad: continue
            chosen   = max(good, key=len)
            rejected = max(bad,  key=len)
            prompt = chat_template_prompt(tok, build_messages(item))
            out.write(json.dumps({"prompt": prompt, "chosen": chosen,
                                    "rejected": rejected, "id": qid}) + "\n")
            n_pairs += 1
    print(f"DPO pairs: {n_pairs} -> {DPO_PAIRS_PATH}")
else:
    print("Skipping DPO-pair build.")

## 10. DPO training on top of SFT (resumable)

In [ ]:
if RUN_DPO:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, PeftModel
    from trl import DPOTrainer, DPOConfig
    from datasets import load_dataset

    assert DPO_PAIRS_PATH.exists(), "Run section 9 first."
    ds = load_dataset("json", data_files=str(DPO_PAIRS_PATH), split="train")

    tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    tok.pad_token = tok.eos_token
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                              bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID, quantization_config=bnb, device_map="auto", trust_remote_code=True,
    )
    if SFT_LORA_DIR.exists():
        policy = PeftModel.from_pretrained(base, str(SFT_LORA_DIR), is_trainable=True)
        peft_cfg = None
    else:
        policy = base
        peft_cfg = LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05, bias="none",
                                task_type="CAUSAL_LM",
                                target_modules=["q_proj","k_proj","v_proj","o_proj",
                                                  "gate_proj","up_proj","down_proj"])

    cfg = DPOConfig(
        output_dir=str(DPO_LORA_DIR),
        per_device_train_batch_size=1, gradient_accumulation_steps=16,
        num_train_epochs=DPO_EPOCHS, learning_rate=DPO_LR, beta=DPO_BETA,
        bf16=True, logging_steps=10,
        save_strategy="steps", save_steps=100, save_total_limit=3,
        max_length=MAX_MODEL_LEN, max_prompt_length=MAX_MODEL_LEN // 2,
        report_to="none", gradient_checkpointing=True, optim="paged_adamw_8bit",
    )
    trainer = DPOTrainer(model=policy, ref_model=None, args=cfg, train_dataset=ds,
                          processing_class=tok, peft_config=peft_cfg)
    resume = _latest_ckpt(DPO_LORA_DIR)
    print("Resuming DPO from", resume) if resume else print("Starting DPO fresh")
    trainer.train(resume_from_checkpoint=resume)
    trainer.save_model(str(DPO_LORA_DIR))
    del trainer, policy, base; gc.collect(); torch.cuda.empty_cache()
else:
    print("Skipping DPO.")

## 11. (Optional) GRPO with the project Judger as outcome reward

Uses `trl.GRPOTrainer`. The reward function returns 1.0 if the completion's extracted answer is judged correct, else 0.0. Resumable via Trainer checkpoints.

In [ ]:
if RUN_GRPO:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, PeftModel
    from trl import GRPOTrainer, GRPOConfig
    from datasets import Dataset

    tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    tok.pad_token = tok.eos_token

    # Build a small prompt dataset (public set) keyed to gold answers.
    rows = []
    for it in public_data:
        prompt = chat_template_prompt(tok, build_messages(it))
        rows.append({"prompt": prompt,
                       "gold": json.dumps(it.get("answer")),
                       "is_mcq": bool(it.get("options"))})
    ds = Dataset.from_list(rows)

    def reward_fn(prompts, completions, **kwargs):
        golds   = kwargs["gold"]
        is_mcqs = kwargs["is_mcq"]
        out = []
        for comp, gold_json, is_mcq in zip(completions, golds, is_mcqs):
            gold = json.loads(gold_json)
            item = {"answer": gold, "options": ["x"] if is_mcq else None}
            out.append(1.0 if judge_response(item, comp) else 0.0)
        return out

    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                              bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID, quantization_config=bnb, device_map="auto", trust_remote_code=True,
    )
    start_adapter = DPO_LORA_DIR if DPO_LORA_DIR.exists() else SFT_LORA_DIR
    if start_adapter.exists():
        policy = PeftModel.from_pretrained(base, str(start_adapter), is_trainable=True)
        peft_cfg = None
    else:
        policy = base
        peft_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                                task_type="CAUSAL_LM",
                                target_modules=["q_proj","k_proj","v_proj","o_proj"])

    cfg = GRPOConfig(
        output_dir=str(GRPO_LORA_DIR),
        per_device_train_batch_size=1, gradient_accumulation_steps=8,
        num_train_epochs=GRPO_EPOCHS, learning_rate=GRPO_LR, beta=GRPO_BETA,
        num_generations=GRPO_NUM_GENERATIONS, max_completion_length=MAX_NEW_TOKENS_SAMPLE,
        max_prompt_length=MAX_MODEL_LEN // 2, temperature=0.9,
        use_vllm=True, vllm_gpu_memory_utilization=0.4,
        bf16=True, logging_steps=5,
        save_strategy="steps", save_steps=50, save_total_limit=3,
        report_to="none", gradient_checkpointing=True, optim="paged_adamw_8bit",
    )
    trainer = GRPOTrainer(model=policy, reward_funcs=reward_fn,
                            args=cfg, train_dataset=ds,
                            processing_class=tok, peft_config=peft_cfg)
    resume = _latest_ckpt(GRPO_LORA_DIR)
    print("Resuming GRPO from", resume) if resume else print("Starting GRPO fresh")
    trainer.train(resume_from_checkpoint=resume)
    trainer.save_model(str(GRPO_LORA_DIR))
    del trainer, policy, base; gc.collect(); torch.cuda.empty_cache()
else:
    print("Skipping GRPO.")

## 12. Merge final adapter (GRPO > DPO > SFT, whichever is latest)

In [ ]:
if RUN_MERGE_FINAL and not MERGED_FINAL_DIR.exists():
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    adapter = (GRPO_LORA_DIR if GRPO_LORA_DIR.exists()
               else DPO_LORA_DIR if DPO_LORA_DIR.exists()
               else SFT_LORA_DIR)
    assert adapter.exists(), "No adapter found."
    print("Merging final adapter:", adapter)
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID, torch_dtype=torch.bfloat16, device_map="cpu", trust_remote_code=True,
    )
    merged = PeftModel.from_pretrained(base, str(adapter)).merge_and_unload()
    merged.save_pretrained(MERGED_FINAL_DIR, safe_serialization=True)
    AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True).save_pretrained(MERGED_FINAL_DIR)
    del base, merged; gc.collect(); torch.cuda.empty_cache()
    print("Merged ->", MERGED_FINAL_DIR)
else:
    print("Merged final:", MERGED_FINAL_DIR, "present" if MERGED_FINAL_DIR.exists() else "skipped")

## 13. Private inference with self-consistency (checkpointed in chunks)

Generates `N_SAMPLES_PER_Q` traces per question, appending to `results/private_samples.jsonl` after every chunk. On resume, questions that already have enough samples are skipped.

In [ ]:
def _load_samples(path: Path):
    by_id = defaultdict(list)
    if not path.exists(): return by_id
    with open(path) as f:
        for line in f:
            try:
                r = json.loads(line); by_id[r["id"]].append(r)
            except Exception: pass
    return by_id

if RUN_PRIVATE_INFER:
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams
    model_path = str(MERGED_FINAL_DIR) if MERGED_FINAL_DIR.exists() else (
        str(MERGED_SFT_DIR) if MERGED_SFT_DIR.exists() else BASE_MODEL_ID)
    use_bnb = (model_path == BASE_MODEL_ID)
    print("Private inference with:", model_path)

    tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    tok.pad_token = tok.eos_token

    by_id = _load_samples(PRIVATE_SAMPLES_PATH)
    todo = [(it, N_SAMPLES_PER_Q - len(by_id[it["id"]])) for it in private_data
             if len(by_id[it["id"]]) < N_SAMPLES_PER_Q]
    print(f"Items needing more samples: {len(todo)}/{len(private_data)}")

    if todo:
        kw = dict(model=model_path, trust_remote_code=True,
                   gpu_memory_utilization=0.85, max_model_len=INFER_MAX_MODEL_LEN,
                   max_num_seqs=256, max_num_batched_tokens=32768,
                   enable_prefix_caching=True)
        if use_bnb: kw.update(quantization="bitsandbytes", load_format="bitsandbytes")
        llm = LLM(**kw)

        # All items in `todo` will get the SAME number of additional samples for batching.
        # We just request N_SAMPLES_PER_Q and trim to needed-many per id.
        sp = SamplingParams(n=max(need for _, need in todo),
                             max_tokens=MAX_NEW_TOKENS_INFER,
                             temperature=INFER_TEMPERATURE, top_p=INFER_TOP_P, top_k=INFER_TOP_K)

        for start in range(0, len(todo), INFER_CHUNK):
            batch = todo[start:start+INFER_CHUNK]
            prompts = [chat_template_prompt(tok, build_messages(it)) for it, _ in batch]
            outs = llm.generate(prompts, sampling_params=sp)
            with open(PRIVATE_SAMPLES_PATH, "a") as f:
                for (item, need), out in zip(batch, outs):
                    base_idx = len(by_id[item["id"]])
                    for j, c in enumerate(out.outputs[:need]):
                        rec = {"id": item["id"], "sample_idx": base_idx + j,
                                "is_mcq": bool(item.get("options")),
                                "trace": c.text.strip()}
                        by_id[item["id"]].append(rec)
                        f.write(json.dumps(rec) + "\n")
            print(f"  flushed {start+len(batch)}/{len(todo)}")
        del llm; gc.collect()
        try:
            import torch; torch.cuda.empty_cache()
        except Exception: pass
else:
    print("Skipping private inference.")

## 14. Vote, then progressive-hint round on low-confidence items (checkpointed)

In [ ]:
by_id = _load_samples(PRIVATE_SAMPLES_PATH)
voted = {}
for item in private_data:
    traces = [r["trace"] for r in by_id.get(item["id"], [])]
    key, frac, best = vote(traces, bool(item.get("options")))
    voted[item["id"]] = {"key": key, "frac": frac, "best": best,
                          "is_mcq": bool(item.get("options"))}
low = [it for it in private_data if voted[it["id"]]["frac"] < PROGRESSIVE_HINT_TAU
       and voted[it["id"]]["key"]]
print(f"Low-confidence (<{PROGRESSIVE_HINT_TAU}): {len(low)}/{len(private_data)}")

if RUN_HINT_ROUND and low:
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams
    model_path = str(MERGED_FINAL_DIR) if MERGED_FINAL_DIR.exists() else (
        str(MERGED_SFT_DIR) if MERGED_SFT_DIR.exists() else BASE_MODEL_ID)
    use_bnb = (model_path == BASE_MODEL_ID)
    tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    tok.pad_token = tok.eos_token

    hint_by_id = _load_samples(PRIVATE_HINTS_PATH)
    todo = [it for it in low if len(hint_by_id[it["id"]]) < N_HINT_SAMPLES]
    print(f"Hint round todo: {len(todo)}")

    if todo:
        kw = dict(model=model_path, trust_remote_code=True,
                   gpu_memory_utilization=0.85, max_model_len=INFER_MAX_MODEL_LEN,
                   max_num_seqs=256, enable_prefix_caching=True)
        if use_bnb: kw.update(quantization="bitsandbytes", load_format="bitsandbytes")
        llm = LLM(**kw)
        sp = SamplingParams(n=N_HINT_SAMPLES, max_tokens=MAX_NEW_TOKENS_INFER,
                             temperature=max(0.4, INFER_TEMPERATURE - 0.2),
                             top_p=INFER_TOP_P, top_k=INFER_TOP_K)
        for start in range(0, len(todo), INFER_CHUNK):
            batch = todo[start:start+INFER_CHUNK]
            prompts = [chat_template_prompt(tok, build_messages(it, hints=[voted[it["id"]]["key"]]))
                       for it in batch]
            outs = llm.generate(prompts, sampling_params=sp)
            with open(PRIVATE_HINTS_PATH, "a") as f:
                for item, out in zip(batch, outs):
                    base_idx = len(hint_by_id[item["id"]])
                    for j, c in enumerate(out.outputs):
                        rec = {"id": item["id"], "sample_idx": base_idx + j,
                                "is_mcq": bool(item.get("options")),
                                "trace": c.text.strip()}
                        hint_by_id[item["id"]].append(rec)
                        f.write(json.dumps(rec) + "\n")
            print(f"  hint flushed {start+len(batch)}/{len(todo)}")
        del llm; gc.collect()
        try:
            import torch; torch.cuda.empty_cache()
        except Exception: pass

# Re-vote, mixing in hint samples for low-confidence items.
hint_by_id = _load_samples(PRIVATE_HINTS_PATH)
for item in private_data:
    combined = [r["trace"] for r in by_id.get(item["id"], [])] + \
                 [r["trace"] for r in hint_by_id.get(item["id"], [])]
    if not combined: continue
    key, frac, best = vote(combined, bool(item.get("options")))
    voted[item["id"]] = {"key": key, "frac": frac, "best": best,
                          "is_mcq": bool(item.get("options"))}
print("Voting refreshed with hint samples.")

## 15. Write `submission.csv`

In [ ]:
def finalize_response(v):
    text, key = v["best"] or "", v["key"]
    if not key: return text if text else "\\boxed{}"
    if v["is_mcq"]:
        if extract_letter(text) == key: return text
    else:
        if normalize_free_answer(text) == key: return text
    return (text.rstrip() + f"\n\nFinal answer: \\boxed{{{key}}}").strip()

with open(SUBMISSION_CSV, "w", newline="") as f:
    w = csv.writer(f, quoting=csv.QUOTE_ALL)
    w.writerow(["id", "response"])
    for item in private_data:
        v = voted.get(item["id"], {"best": "", "key": "", "is_mcq": bool(item.get("options"))})
        w.writerow([item["id"], finalize_response(v)])
print(f"Wrote {SUBMISSION_CSV} with {len(private_data)} rows.")

## 16. Public-set accuracy sanity check (optional)

In [ ]:
RUN_PUBLIC_EVAL = True
if RUN_PUBLIC_EVAL:
    bucket = defaultdict(list)
    if SELF_SAMPLES_PATH.exists():
        with open(SELF_SAMPLES_PATH) as f:
            for line in f:
                r = json.loads(line); bucket[r["id"]].append(r["trace"])
    n_ok = 0; n_done = 0
    for it in public_data:
        traces = bucket.get(it["id"], [])
        if not traces: continue
        key, frac, best = vote(traces, bool(it.get("options")))
        synth = finalize_response({"best": best, "key": key, "is_mcq": bool(it.get("options"))})
        n_ok += int(judge_response(it, synth)); n_done += 1
    print(f"Public-set accuracy (from cached self-samples): {n_ok}/{n_done} = {100*n_ok/max(1,n_done):.2f}%")
else:
    print("Set RUN_PUBLIC_EVAL=True to estimate.")